# 01 - Data Preparation

**Project:** E-Commerce Order Delivery Time & Transit Delay Estimator
**Dataset:** Olist Brazilian E-Commerce (`olistbr/brazilian-ecommerce`)
**Plan:** `data-preparation.md`

Turns six raw tables into one table with one row per shipment (an order from one seller).

| Column | Meaning |
|---|---|
| `order_id`, `seller_id` | identifiers, kept for traceability, not features |
| `dispatch_date` | date the carrier received the package |
| `weight_kg` | total package weight |
| `distance_km` | straight-line seller-to-customer distance |
| `warehouse_load` | orders the same seller handed to the carrier on the same day |
| `carrier_tier` | Economy / Standard / Express, a proxy from freight cost |
| `transit_days` | **target**: carrier pickup to customer delivery |

No EDA and no modelling in this notebook.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Works locally (run from notebooks/ or the project root) and in Colab (run from /content).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

TRANSIT_CAP_DAYS = 60            # starting point, revisit in EDA
BRAZIL_LAT = (-34.0, 6.0)        # rough bounding box, used to drop bad geolocation points
BRAZIL_LNG = (-74.0, -34.0)

row_log = []                     # (step, rows) so we can see where data is lost


def log(step, df):
    row_log.append((step, len(df)))
    print(f"{step:<45}{len(df):>9,}")

### Raw files

If `data/raw/` is empty (for example in a fresh Colab session) the files are downloaded with `kagglehub`.
That needs a Kaggle token in the `KAGGLE_API_TOKEN` environment variable. In Colab, store it under *Secrets* and export it to `os.environ` first. Never paste the token into the notebook.

In [ ]:
REQUIRED = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
]

if any(not (RAW / f).exists() for f in REQUIRED):
    import shutil

    import kagglehub

    src = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    for f in src.glob("*.csv"):
        shutil.copy2(f, RAW / f.name)

missing = [f for f in REQUIRED if not (RAW / f).exists()]
assert not missing, f"missing raw files: {missing}"
print("raw files ready in", RAW)

## 1. Select orders and compute the target

Keep delivered orders that have both the carrier-pickup and the customer-delivery timestamp.
`transit_days` is the shipping leg only, so approval and seller handling time are left out.

In [ ]:
orders = pd.read_csv(
    RAW / "olist_orders_dataset.csv",
    parse_dates=["order_delivered_carrier_date", "order_delivered_customer_date"],
)
log("orders (raw)", orders)

orders = orders[orders["order_status"] == "delivered"]
log("delivered", orders)

orders = orders.dropna(subset=["order_delivered_carrier_date", "order_delivered_customer_date"]).copy()
log("delivered with both timestamps", orders)

orders["dispatch_date"] = orders["order_delivered_carrier_date"].dt.normalize()
orders["transit_days"] = (
    orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400
orders = orders[["order_id", "customer_id", "dispatch_date", "transit_days"]]

orders["transit_days"].describe().round(2)

## 2. One row per shipment, weight and warehouse load

Group the item rows by `order_id` and `seller_id`, summing weight and freight.
If any product in a shipment has no weight, the whole shipment weight is set to missing instead of being under-counted.

`warehouse_load` is the number of shipments the same seller handed to the carrier on the same calendar day.
It is counted **before** multi-seller orders are removed, because those orders were still real work for the seller.

In [ ]:
items = pd.read_csv(
    RAW / "olist_order_items_dataset.csv",
    usecols=["order_id", "product_id", "seller_id", "freight_value"],
)
products = pd.read_csv(RAW / "olist_products_dataset.csv", usecols=["product_id", "product_weight_g"])

items = items.merge(products, on="product_id", how="left")
items = items[items["order_id"].isin(orders["order_id"])].copy()
items["weight_missing"] = items["product_weight_g"].isna()

shipments = items.groupby(["order_id", "seller_id"], as_index=False).agg(
    weight_g=("product_weight_g", "sum"),
    weight_missing=("weight_missing", "any"),
    freight_value=("freight_value", "sum"),
)
shipments.loc[shipments["weight_missing"], "weight_g"] = np.nan
shipments["weight_kg"] = shipments["weight_g"] / 1000

shipments = shipments.merge(orders, on="order_id", how="inner")
log("shipments (order x seller)", shipments)

# Warehouse load: shipments of the same seller on the same dispatch day (including this one).
shipments["warehouse_load"] = shipments.groupby(["seller_id", "dispatch_date"])["order_id"].transform("nunique")

# One origin per package: drop orders that ship from more than one seller.
n_sellers = shipments.groupby("order_id")["seller_id"].transform("nunique")
shipments = shipments[n_sellers == 1].copy()
log("single-seller orders only", shipments)

shipments = shipments[shipments["weight_kg"].notna() & (shipments["weight_kg"] > 0)].copy()
log("valid weight (> 0)", shipments)

## 3. Distance

Geolocation has many points per zip prefix. Points outside Brazil's bounding box are removed,
then one point per prefix is taken as the **median** latitude and longitude.
Distance is the straight-line (haversine) distance between the seller's and the customer's zip prefix.

In [ ]:
geo = pd.read_csv(
    RAW / "olist_geolocation_dataset.csv",
    usecols=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
)
in_brazil = geo["geolocation_lat"].between(*BRAZIL_LAT) & geo["geolocation_lng"].between(*BRAZIL_LNG)
print(f"geolocation points outside Brazil box: {(~in_brazil).sum():,}")

zip_xy = (
    geo[in_brazil]
    .groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .median()
    .rename(columns={"geolocation_lat": "lat", "geolocation_lng": "lng"})
)
print(f"zip prefixes with a coordinate: {len(zip_xy):,}")

customers = pd.read_csv(RAW / "olist_customers_dataset.csv", usecols=["customer_id", "customer_zip_code_prefix"])
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv", usecols=["seller_id", "seller_zip_code_prefix"])

shipments = (
    shipments.merge(customers, on="customer_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(zip_xy.add_prefix("seller_"), left_on="seller_zip_code_prefix", right_index=True, how="left")
    .merge(zip_xy.add_prefix("customer_"), left_on="customer_zip_code_prefix", right_index=True, how="left")
)


def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = (np.radians(x) for x in (lat1, lng1, lat2, lng2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


shipments["distance_km"] = haversine_km(
    shipments["seller_lat"], shipments["seller_lng"], shipments["customer_lat"], shipments["customer_lng"]
)
shipments = shipments[shipments["distance_km"].notna()].copy()
log("distance available", shipments)

## 4. Carrier tier (proxy)

Olist has no carrier field. The closest real signal of service level is freight price per kg per km,
split into three equal-sized bands. Distance is floored at 1 km so same-area shipments do not divide by zero.

The tier uses only weight, distance and freight, never the target. It is a proxy, and because freight depends on
weight and distance it overlaps with those two features.

In [ ]:
rate = shipments["freight_value"] / (shipments["weight_kg"] * shipments["distance_km"].clip(lower=1))
shipments["carrier_tier"] = pd.qcut(rate.rank(method="first"), 3, labels=["Economy", "Standard", "Express"])

shipments.groupby("carrier_tier", observed=True).agg(
    shipments=("order_id", "size"),
    median_rate=("freight_value", lambda s: float(np.median(rate[s.index]))),
)

## 5. Quality filters and save

`shipments_full.csv` keeps every derived column **before** the transit-time filters, so EDA can change them.
`shipments.csv` is the cleaned file used for modelling.

In [ ]:
COLUMNS = [
    "order_id", "seller_id", "dispatch_date",
    "weight_kg", "distance_km", "warehouse_load", "carrier_tier", "transit_days",
]
shipments_full = shipments[COLUMNS].sort_values(["dispatch_date", "order_id"]).reset_index(drop=True)
log("shipments_full (before transit filters)", shipments_full)

final = shipments_full[shipments_full["transit_days"] >= 0]
log("transit_days >= 0", final)

final = final[final["transit_days"] <= TRANSIT_CAP_DAYS]
log(f"transit_days <= {TRANSIT_CAP_DAYS}", final)

final = final.drop_duplicates(["order_id", "seller_id"]).reset_index(drop=True)
log("no duplicate order x seller", final)

shipments_full.to_csv(OUT / "shipments_full.csv", index=False)
final.to_csv(OUT / "shipments.csv", index=False)
print("saved:", OUT / "shipments_full.csv", "and", OUT / "shipments.csv")

## 6. Checks

In [ ]:
assert list(final.columns) == COLUMNS, "unexpected columns"
assert final[COLUMNS].notna().all().all(), "nulls found"
assert (final["transit_days"] > 0).all() and (final["transit_days"] <= TRANSIT_CAP_DAYS).all()
assert final["distance_km"].between(0, 4500).all(), "distance outside plausible range for Brazil"
assert (final["warehouse_load"] >= 1).all()
assert final["carrier_tier"].nunique() == 3
assert not final.duplicated(["order_id", "seller_id"]).any()

print("all checks passed\n")
print("rows:", f"{len(final):,}", "| dispatch dates:", final["dispatch_date"].min(), "->", final["dispatch_date"].max())
print("\ncarrier_tier share:\n", final["carrier_tier"].value_counts(normalize=True).round(3).to_string())
print("\nwarehouse_load share (top values):\n", final["warehouse_load"].value_counts(normalize=True).head(5).round(3).to_string())
final[["weight_kg", "distance_km", "warehouse_load", "transit_days"]].describe().round(2)